### 构建聊天机器人界面：文本或语音输入、多 LLM 支持、记忆持久化

本笔记本对应第 2 周常见练习方向：用 **Gradio** 快速搭一个可切换模型的客服/助手原型，并可选保留对话历史（跨模型共享记忆）。


## 练习目标（理念）

用 Gradio 做一个友好的聊天原型，能力包括：

- **多语言模型**：对话中随时切换（OpenAI / Anthropic Claude / 本地 LLaMA via Ollama）
- **可选记忆持久化**：把聊天历史存起来再转发给当前模型，切换模型时仍能「接着聊」
- **语音输入**：Python `speech_recognition` + `sounddevice` 录音，再转成文本塞进输入框

## 技术栈对照

| 组件 | 本练习里的用法 |
|------|----------------|
| OpenAI API | `OpenAI().chat.completions.create` |
| Anthropic Claude | `anthropic.Anthropic().messages.create` |
| Ollama（本地） | `requests.post` 打 `/api/chat` |
| 语音转写 | `speech_recognition` + Google recognizer |
| UI | `gr.Blocks` + Checkbox / Dropdown / Textbox |

## 怎么跑

1. 安装依赖：`gradio`、`openai`、`anthropic`、`speech_recognition`、`sounddevice`、`numpy`、`python-dotenv` 等
2. `.env` 配置 `OPENAI_API_KEY`、`ANTHROPIC_API_KEY`（按需）；本地模型需 Ollama 在跑
3. 从上到下运行；最后一格 `demo.launch()` 打开界面
4. 可打字发送，或点录音按钮 → 停止后转写进 ChatBox → Send

> 补充：部分云端 API 已支持直接音频输入；本练习仍用独立转写模块，便于看清「录音 → 文本 → LLM」链路。


In [37]:
# ========== 导入：环境变量、HTTP、OpenAI、Anthropic ==========

# 导入标准库 os：读 API Key 等环境变量
import os
# 导入 requests：用 HTTP POST 调本地 Ollama /api/chat
import requests
# 从 dotenv 导入 load_dotenv：加载 .env，避免密钥写进笔记本
from dotenv import load_dotenv
# 从 openai 导入 OpenAI：云端 Chat Completions
from openai import OpenAI
# 导入 anthropic：官方 SDK，调 Claude Messages API
import anthropic


In [38]:
# ========== 导入：语音录制与识别相关库 ==========

# speech_recognition：把 AudioData 交给识别引擎（这里用 Google）
import speech_recognition as sr
# sounddevice：从麦克风采 PCM 音频流
import sounddevice as sd
# numpy：拼接录音 buffer、转 bytes
import numpy as np


In [39]:
# ========== 导入：Gradio，用于图形界面原型 ==========

# gradio 别名 gr：Blocks / Button / Textbox / Dropdown 等组件
import gradio as gr


In [40]:
# ========== 录音基础设施：全局 buffer + 输入流回调 ==========

# buffer：临时存放每一帧麦克风数据（NumPy 数组列表）
buffer = [] # For temporarily holding sound recording

# sounddevice 回调：每次来一块 indata 就拷贝进 buffer（避免底层复用内存被覆盖）
def callback(indata, frames, time, status):
    buffer.append(indata.copy())

# 打开输入流：16kHz 单声道 int16，与后面 AudioData 参数一致；先不 start，等按钮切换
stream = sd.InputStream(callback=callback, samplerate=16000, channels=1, dtype='int16')


In [41]:
# ========== 录音开关 + Google 语音转写 ==========

# 处理录制状态：False=空闲可开始；True=正在录，再点则停止并转写
def toggle_recording(state):
    global stream, buffer
    # 调试：打印当前 state
    print('state', state)

    if not state:
        # 开始录音：清空旧 buffer，启动流；按钮文案改为 Stop
        buffer.clear()
        stream.start()
        return gr.update(value="Stop Recording"), 'Recording...', not state
    else:
        # 停止录音：停流 → 拼接全部帧 → 转写 → 文本填回输入框
        stream.stop()
        audio = np.concatenate(buffer, axis=0)
        text = transcribe(audio)
        return gr.update(value="Start Recording"), text, not state

# 通过 Google 语音识别把录音转成文本
def transcribe(recording, sample_rate=16000):
    r = sr.Recognizer()

    # 将 NumPy int16 数组包装成 speech_recognition.AudioData
    audio_data = sr.AudioData(
    recording.tobytes(),              # Raw byte data
    sample_rate,                     # Sample rate
        2                                # Sample width in bytes (16-bit = 2 bytes)
    )

    # recognize_google：在线识别（需网络）；返回字符串
    text = r.recognize_google(audio_data)
    print("You said:", text)
    return text


### LLM 与 API 设置

接下来加载密钥，并实现一个统一的 `LLMHandler`：按模型名把请求路由到 OpenAI / Claude / Ollama。


##### 从 `.env` 加载 API 密钥

用 `load_dotenv(override=True)` 读入环境变量；下面只打印密钥前缀，方便确认配置是否生效。


In [42]:
# ========== 加载环境变量并打印密钥前缀（便于调试） ==========

# override=True：.env 覆盖已有同名环境变量
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

# 存在则打印前几位；不存在则明确提示未设置（打印文案保持原样）
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")


OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key not set


### 用类封装 API 调用：按所选模型路由请求

`LLMHandler` 维护 `message_history`，并提供 `llm_call` / `call_openai` / `call_claude` / `call_ollama`。


In [43]:
# ========== LLMHandler：统一封装多后端 + 可选记忆持久化 ==========

class LLMHandler:
    def __init__(self, system_message: str = '', ollama_api:str='http://localhost:11434/api/chat'):
        # 未传入则用默认 system；英文原文影响模型行为，保持不翻译
        self.system_message = system_message if system_message else "You are a helpful assistant. Always reply in Markdown"
        # 对话历史：OpenAI/Ollama 风格的 role/content 列表
        self.message_history = []

        # 初始化各后端客户端；Ollama 用原生 /api/chat URL + JSON Content-Type
        self.openai = OpenAI()
        self.claude = anthropic.Anthropic()
        self.OLLAMA_API = ollama_api
        self.OLLAMA_HEADERS = {"Content-Type": "application/json"}

    def llm_call(self, model: str = 'gpt-4o-mini', prompt: str = '', memory_persistence=True):
        # 未选模型直接返回提示（文案保持原样）
        if not model:
            return 'No model specified'

        # 首轮且非 claude：带 system 的完整模板；否则只追加 user
        # （Claude 的 system 走独立参数，见 call_claude）
        message = self.get_message_template(prompt, initial=True) if (
            not self.message_history and not 'claude' in model
             ) else self.get_message_template(prompt)

        # 记忆开：把本轮 message 接到历史；关：历史重置为本轮
        if memory_persistence:
            self.message_history.extend(message)
        else:
            self.message_history = message

        # 按模型名子串路由到不同后端
        try:
            if 'gpt' in model:
                response = self.call_openai(model=model)
            elif 'claude' in model:
                response = self.call_claude(model=model)
            elif 'llama' in model:
                response = self.call_ollama(model=model)
            else:
                response = f'{model.title()} is not supported or not a valid model name.'
        except Exception as e:
            response = f'Failed to retrieve response. Reason: {e}'

        # 记忆开：把助手回复也写回历史，供下一轮接着聊
        if memory_persistence:
            self.message_history.append({
                "role": "assistant",
                "content": response
            })

        return response

    def get_message_template(self, prompt: str = '', initial=False):
        # initial=True：system + user；否则只有 user（接在已有历史上）
        initial_template = [
            {"role": "system", "content": self.system_message},
            {"role": "user", "content": prompt}
        ]
        general_template = [
            {"role": "user", "content": prompt}
        ]
        return initial_template if initial else general_template

    def call_openai(self, model: str = 'gpt-4o-mini'):
        # Chat Completions：messages 用整段 message_history
        completion = self.openai.chat.completions.create(
            model=model,
            messages=self.message_history,
        )
        response = completion.choices[0].message.content
        return response

    def call_ollama(self, model: str = "llama3.2"):
        # 原生 Ollama /api/chat；stream=False 一次拿完整 JSON
        payload = {
            "model": model,
            "messages": self.message_history,
            "stream": False
        }

        response = requests.post(url=self.OLLAMA_API, headers=self.OLLAMA_HEADERS, json=payload)
        return response.json()["message"]["content"]

    def call_claude(self, model: str = "claude-3-haiku-20240307"):
        # Anthropic Messages API：system 单独传；messages 用历史（通常不含 system role）
        message = self.claude.messages.create(
            model=model,
            system=self.system_message,
            messages=self.message_history,
            max_tokens=500
        )
        # content 是块列表，取第一块的 text
        response = message.content[0].text
        return response


In [44]:
# ========== Gradio 回调包装：单例 handler + 发送后清空输入框 ==========

# 全局一个 LLMHandler，跨多次点击共享 message_history（配合 memory checkbox）
llm_handler = LLMHandler()

# 界面收到用户提示时调用：返回 (回复文本, 空字符串清 ChatBox)
def llm_call(model, prompt, memory_persistence):
    response = llm_handler.llm_call(model=model, prompt=prompt, memory_persistence=memory_persistence)
    return response, ''


In [45]:
# ========== 下拉可选模型名列表（字符串必须与路由逻辑 / 后端一致） ==========

AVAILABLE_MODELS = ["gpt-4", "gpt-3.5", "claude-3-haiku-20240307", "llama3.2", "gpt-4o-mini"]


In [46]:
# ========== Gradio Blocks：录音状态 + 模型/记忆开关 + 发送 ==========

with gr.Blocks() as demo:
    # 录音开关状态：False=未录，True=录制中（与 toggle_recording 联动）
    state = gr.State(False) # Recording state (on/off)
    with gr.Row():
        
        with gr.Column():
            # 展示模型回复（Markdown）
            out = gr.Markdown(label='Message history')
            with gr.Row():
                # 是否把历史继续喂给模型（跨模型共享记忆）
                memory = gr.Checkbox(label='Toggle memory', value=True) # Handle memory status (on/off) btn
                # 模型下拉：choices 来自 AVAILABLE_MODELS
                model_choice = gr.Dropdown(label='Model', choices=AVAILABLE_MODELS, interactive=True) # Model selection dropdown
            # 文本输入；录音转写结果也会写回这里
            query_box = gr.Textbox(label='ChatBox', placeholder="Your message")
            # 录音按钮：文案由 toggle_recording 在 Start/Stop 间切换
            record_btn = gr.Button(value='Record voice message') # Start/stop recording btn
            send_btn = gr.Button("Send") # Send prompt btn
      
            
    
    # 点录音：输入 state → 更新按钮文案、输入框文本、翻转 state
    record_btn.click(fn=toggle_recording, inputs=state, outputs=[record_btn, query_box, state])
    # 点发送：选中模型 + 输入框 + memory → 回复到 out，并清空 query_box
    send_btn.click(fn=llm_call, inputs=[model_choice, query_box, memory], outputs=[out, query_box])
    

# 启动 Gradio 服务（默认本地端口）
demo.launch()


* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
